# VisionOps — real NVIDIA GPU TensorRT verification

**Status: EXECUTED ON REAL NVIDIA HARDWARE — this notebook is the reproducible recipe.**
It was run on a Tesla T4 (driver 580.82.07, CUDA 12.8, TensorRT 11.3.0.99) and the
authoritative results are committed under `docs/evidence/tensorrt/final/`. Re-running it
reproduces them here. It never reports the machine it happens to sit on as TensorRT
capable: every number it prints is measured in this session, or it prints
`NOT VERIFIED`.

Your only actions: **open this notebook, select a GPU runtime, upload
`visionops-gpu-bundle.zip`, Run all.** Everything after that is automatic.

What it does, in order:

1. inspects the session (`nvidia-smi`, `python`, `pip show torch`) **before installing anything**,
2. installs only what the harness needs, and verifies TensorRT by creating a real `Builder`
   (a successful `import` is not accepted as proof),
3. locates the uploaded bundle and resolves every path from its `MANIFEST.json` — no
   Windows or hardcoded paths,
4. checks the ONNX SHA-256 against both the bundle manifest and the qualified contract
   record before any inference happens,
5. runs `scripts.verify_tensorrt_gpu` on the qualified ONNX for **FP32**: real inference on
   real frames, parity against a reference runtime through the **same canonical decoder**,
   CUDA-synchronised latency and throughput,
6. builds a true **mixed-FP16** graph with `scripts.convert_fp16_onnx.py` (ModelOpt AutoCast
   plus Ultralytics metadata restoration), checks FP32-vs-FP16 ONNX parity on the same
   samples, then runs the FP16 engine and confirms the engine itself declares `HALF`,
7. writes the JSON evidence plus `GPU_RUN_RESULT.md` for both runs, merges them under
   `final/fp32` and `final/true_fp16`, and packages everything as
   `visionops-tensorrt-evidence.zip` for download.

With no GPU or no TensorRT the harness exits `BLOCKED_BY_NVIDIA_HARDWARE` (code 3) and
writes `blocked.json` — it never benchmarks the CPU and calls that a TensorRT result.

Runtimes: **Colab** — `Runtime → Change runtime type → T4 GPU`. **Kaggle** — see
`code/docs/evidence/tensorrt/KAGGLE_GPU_RUN.md`, or just Run all here (this notebook
detects both hosts).

## 1. Session facts — measured before anything is installed

The point of doing this first is diagnosis: a CUDA/TensorRT mismatch is much easier to
read from the image's own versions than from a failed install.

In [ ]:
import platform, shutil, subprocess, sys

def run(command):
    """Run a command and return its output; never raise on a missing tool."""
    try:
        done = subprocess.run(command, capture_output=True, text=True, timeout=180)
        return (done.stdout or done.stderr).strip()
    except Exception as error:
        return f'unavailable: {type(error).__name__}: {error}'

print('python      ', sys.version.split()[0])
print('platform    ', platform.platform())
print('nvidia-smi  ', shutil.which('nvidia-smi') or 'NOT FOUND')
print('\n--- nvidia-smi ---')
print(run(['nvidia-smi']))
print('\n--- nvidia-smi (query-gpu) ---')
print(run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
           '--format=csv,noheader']))
print('\n--- pip show torch (before any install) ---')
print(run([sys.executable, '-m', 'pip', 'show', 'torch']))

try:
    import torch
except Exception as error:
    raise SystemExit(f'torch is not importable in this image: {error}')

print('\ntorch       ', torch.__version__, '| cuda build:', torch.version.cuda)
print('cuda avail  ', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'NO NVIDIA GPU: Colab -> Runtime -> Change runtime type -> T4 GPU. '
    'Kaggle -> Settings -> Accelerator -> GPU T4 x2. Then Run all again.')
properties = torch.cuda.get_device_properties(0)
print('device      ', torch.cuda.get_device_name(0))
print('vram GiB    ', round(properties.total_memory / 2 ** 30, 1))
print('compute cap ', f'{properties.major}.{properties.minor}')
print('fast fp16   ', torch.cuda.get_device_capability(0) >= (5, 3))

## 2. Dependencies — inspect first, one retry, then stop

`torch` is deliberately **not** installed or upgraded: Colab and Kaggle already ship a
CUDA-matched build, and replacing it is the fastest way to break the session. TensorRT is
driven through its Python API because `trtexec` is not shipped in the pip wheels.

In [ ]:
import importlib, subprocess, sys

INSTALL_LOG = []

def has(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False

def attempt(label, packages):
    """One pip attempt, with the full error preserved for the evidence."""
    command = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    done = subprocess.run(command, capture_output=True, text=True)
    INSTALL_LOG.append({'attempt': label, 'command': ' '.join(command),
                        'returncode': done.returncode,
                        'stderr_tail': (done.stderr or '')[-1500:]})
    print(f'[{label}] returncode={done.returncode}')
    if done.returncode != 0:
        print((done.stderr or '')[-900:])
    return done.returncode == 0

NEEDED = {'onnx': 'onnx', 'supervision': 'supervision', 'opencv-python-headless': 'cv2',
          'numpy': 'numpy'}
missing = [package for package, module in NEEDED.items() if not has(module)]
if missing:
    attempt('base', missing)
for package, module in NEEDED.items():
    print(f'{package:<26}{has(module)}')

# TensorRT: PyPI first, then exactly one alternate if the wheel does not match the image.
if not has('tensorrt'):
    attempt('tensorrt-pypi', ['tensorrt'])
if not has('tensorrt'):
    attempt('tensorrt-cuda-major', ['tensorrt>=10.0,<11'])
if has('tensorrt'):
    import tensorrt as trt
    builder = trt.Builder(trt.Logger(trt.Logger.WARNING))  # a real builder, not just an import
    print('tensorrt    ', trt.__version__, '| builder created:', builder is not None)
    print('trt10 api   ', 'execute_async_v3:', hasattr(trt.IExecutionContext, 'execute_async_v3'),
          '| set_tensor_address:', hasattr(trt.IExecutionContext, 'set_tensor_address'),
          '| num_io_tensors:', hasattr(trt.ICudaEngine, 'num_io_tensors'))
else:
    print('TENSORRT UNAVAILABLE. The harness will exit BLOCKED_BY_NVIDIA_HARDWARE (code 3) '
          'rather than benchmark the CPU. Install attempts are preserved in INSTALL_LOG; '
          'see code/docs/evidence/tensorrt/KAGGLE_GPU_RUN.md step 4 for the alternates.')

# nvidia-modelopt produces the mixed-precision graph that carries FP16 on TensorRT 11+,
# which removed BuilderFlag.FP16. A failure here is recorded, not fatal: the FP32 run
# still proceeds and the FP16 precision is then reported NOT VERIFIED with the reason.
if not has('modelopt'):
    attempt('nvidia-modelopt', ['nvidia-modelopt'])
print('nvidia-modelopt          ', has('modelopt'))

try:
    import onnxruntime as ort
    print('onnxruntime ', ort.__version__, ort.get_available_providers())
except Exception as error:
    print('onnxruntime unavailable:', error)

## 3. Inputs — locate the uploaded bundle

Upload `visionops-gpu-bundle.zip` when prompted (that is the one manual step). Kaggle
auto-extracts a dataset, so both an extracted `MANIFEST.json` and a raw `.zip` are looked
for, in both hosts' input directories. Every path below is resolved from the manifest.

In [ ]:
import hashlib, json, pathlib, shutil, subprocess, zipfile

IN_COLAB = pathlib.Path('/content').exists() and not pathlib.Path('/kaggle').exists()
WORK = pathlib.Path('/content/visionops') if IN_COLAB else pathlib.Path('/kaggle/working/visionops')
WORK.mkdir(parents=True, exist_ok=True)
SEARCH = [pathlib.Path.cwd(), pathlib.Path('/content'), pathlib.Path('/kaggle/input'),
          pathlib.Path('/kaggle/working')]
print('host        ', 'colab' if IN_COLAB else 'kaggle/other', '| working dir', WORK)

def locate(pattern):
    for root in SEARCH:
        if root.exists():
            hits = sorted(root.glob(f'**/{pattern}'))
            if hits:
                return hits[0]
    return None

found = locate('MANIFEST.json')
if found is None:
    archive = locate('visionops-gpu-bundle.zip')
    if archive is None and IN_COLAB:
        print('Upload visionops-gpu-bundle.zip (Python 3.12 build of make_gpu_bundle) ...')
        from google.colab import files
        uploaded = files.upload()
        archive = pathlib.Path.cwd() / next(iter(uploaded))
    if archive is not None and archive.is_file():
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(WORK)
        print('extracted   ', archive, '->', WORK)
        found = locate('MANIFEST.json')

BUNDLE = found.parent if found is not None else None
if BUNDLE is not None:
    manifest = json.loads((BUNDLE / 'MANIFEST.json').read_text())
    CODE = BUNDLE / 'code'
    ONNX = BUNDLE / manifest['onnx']['archive_path']
    CONTRACT = BUNDLE / manifest['contract_record']['archive_path']
    SAMPLES = BUNDLE / 'samples'
else:
    # Fallback: the code is public, the 12 MB artifact is not (var/ is gitignored).
    REPO = 'https://github.com/Shivakushwah143/-VISIONOPS.git'
    if not (WORK / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO, str(WORK)], check=True)
    CODE = WORK
    ONNX = WORK / 'var/model/hansung-p3.onnx'
    CONTRACT = WORK / 'var/model/hansung-p3.json'
    SAMPLES = None
    manifest = None
    print('no bundle found: cloned the repo. The ONNX artifact and its contract record must '
          'be uploaded into var/model/ before this can run.')

print('code        ', CODE, CODE.is_dir())
print('onnx        ', ONNX, ONNX.is_file(), ONNX.stat().st_size if ONNX.is_file() else '')
print('contract    ', CONTRACT, CONTRACT.is_file())
print('samples     ', SAMPLES, len(list(SAMPLES.glob('*.jpg'))) if SAMPLES else 'n/a')

In [ ]:
# Identity gate: the harness refuses a mismatched artifact anyway, but failing here means
# a wrong upload is caught before a single engine build is spent on it.
def sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

actual = sha256(ONNX)
qualified = json.loads(CONTRACT.read_text())['artifact']['sha256']
print('onnx sha256     ', actual)
print('qualified record', qualified)
if manifest:
    print('bundle manifest ', manifest['onnx']['sha256'],
          '| match:', manifest['onnx']['sha256'] == actual)
assert actual == qualified, 'the ONNX does not match the qualified contract record: wrong upload'
print('artifact identity OK; git commit of the bundle:',
      (manifest or {}).get('git_commit'), (manifest or {}).get('working_tree'))

## 4. Run the real verification

Two runs, both against the qualified ONNX, both through the same canonical decoder and the
same evaluation samples:

1. **FP32** - the qualified graph, unchanged.
2. **true mixed FP16** - `scripts.convert_fp16_onnx.py` builds a mixed-precision graph with
   ModelOpt AutoCast and restores the Ultralytics metadata AutoCast drops; the FP32 vs
   FP16 ONNX parity check runs on the same samples; only then is the FP16 engine built,
   and the engine must declare `HALF` for the precision to count.

INT8 is not attempted: it requires a representative calibration set and a parity
validation pass that this repository does not have.

In [ ]:
import subprocess, sys

import os, shutil

EVIDENCE = CODE / 'docs/evidence/tensorrt'
SAMPLE_ARGS = ['--samples-dir', str(SAMPLES)] if SAMPLES and SAMPLES.is_dir() else []
if not SAMPLE_ARGS:
    raise SystemExit('no evaluation samples: the bundle carries samples/ (real frames cut from '
                     'the local evaluation video var/media/ppe-2.mp4, which is gitignored) and is '
                     'the intended input path')

COMMON = ['--contract-record', str(CONTRACT), *SAMPLE_ARGS, '--warmup', '20', '--iterations', '100']

def verify(arguments, label):
    command = [sys.executable, '-m', 'scripts.verify_tensorrt_gpu', *arguments, *COMMON]
    print('\n$ ' + ' '.join(command), flush=True)
    done = subprocess.run(command, cwd=str(CODE), text=True)
    print(f'[{label}] exit code {done.returncode} '
          '(0 = verified, 3 = blocked, 4 = ran but failed)')
    return done.returncode

# ---- FP32: the qualified ONNX, unchanged ---------------------------------------
FP32_DIR, FP16_DIR = EVIDENCE / 'fp32', EVIDENCE / 'true_fp16'
for stale in (FP32_DIR, FP16_DIR):
    shutil.rmtree(stale, ignore_errors=True)   # a rerun must not mix with an older run
fp32_exit = verify(['--onnx', str(ONNX), '--precisions', 'fp32', '--out', str(FP32_DIR)], 'fp32')

# ---- true mixed FP16 ------------------------------------------------------------
# TensorRT 11 removed BuilderFlag.FP16, so precision must live in the graph. ModelOpt
# AutoCast builds it but DROPS the Ultralytics metadata (including `names`), which
# edge.pipeline.Detector rejects as model_missing_class_metadata. The converter restores
# it and verifies the taxonomy is identical instead of bypassing that check.
FP16_ONNX = CODE / 'var/model/hansung-p3-fp16.onnx'
CONVERSION_REPORT = FP32_DIR / 'fp32_onnx_to_mixed_fp16_onnx.conversion.json'
FP32_DIR.mkdir(parents=True, exist_ok=True)
convert = subprocess.run(
    [sys.executable, '-m', 'scripts.convert_fp16_onnx', '--onnx', str(ONNX),
     '--out', str(FP16_ONNX), '--report', str(CONVERSION_REPORT)],
    cwd=str(CODE), text=True)
print('[convert] exit code', convert.returncode, '(0 = converted, 2 = tool unavailable)')

fp16_exit = convert.returncode
if convert.returncode == 0:
    fp16_exit = verify(['--onnx', str(ONNX), '--fp16-onnx', str(FP16_ONNX),
                        '--precisions', 'fp16', '--out', str(FP16_DIR)], 'true-fp16')
else:
    print('no FP16 graph was produced, so there is no true FP16 result. '
          'The FP32 run is unaffected; FP16 stays NOT VERIFIED rather than being measured '
          'under a label the engine cannot back.')

# ---- merge both runs into the repository's final layout -------------------------
FINAL = EVIDENCE / 'final'
FINAL.mkdir(parents=True, exist_ok=True)
for source_dir, name in ((FP32_DIR, 'fp32'), (FP16_DIR, 'true_fp16')):
    if source_dir.is_dir():
        shutil.copytree(source_dir, FINAL / name, dirs_exist_ok=True)
print('final evidence:', sorted(str(path.relative_to(EVIDENCE)) for path in FINAL.rglob('*.json')))
print({'fp32_exit': fp32_exit, 'fp16_exit': fp16_exit})

## 5. Read the evidence the run just produced

Nothing is summarised from memory here: every line is read back out of the JSON the run
wrote, so a missing measurement shows up as `NOT VERIFIED` instead of a blank.

In [ ]:
import json

def find(*parts):
    # Evidence may sit at the top level or under final/fp32 or final/true_fp16.
    for base in (EVIDENCE / 'true_fp16', EVIDENCE / 'fp32', EVIDENCE / 'final' / 'true_fp16',
                 EVIDENCE / 'final' / 'fp32', EVIDENCE):
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate
    return None

summary_path = find('tensorrt-verification.json')
blocked_path = find('blocked.json')
if summary_path:
    summary = json.loads(summary_path.read_text())
    environment = summary['environment']
    print('STATUS      ', summary['status'])
    print('GPU         ', environment.get('torch_device_name'), environment.get('gpu_names'))
    print('driver/cuda ', environment.get('driver'), '/', environment.get('cuda'),
          '| tensorrt', environment.get('tensorrt'))
    print('onnx sha256 ', summary['model']['sha256'])
    print('mapping ver ', summary['class_mapping_version'],
          '| head width', summary['contract'].get('source_count'))
    print('\nbenchmark (model-only FPS excludes decode/tracking/events):')
    for row in summary['benchmark_table']:
        print('  ', ' | '.join(str(row.get(key, 'NOT VERIFIED')) for key in
              ('runtime', 'precision', 'p50_ms', 'p95_ms', 'mean_ms', 'throughput_fps_model_only')))
    print('\nparity:')
    for precision, result in sorted(summary['precisions'].items()):
        parity = result.get('parity')
        if parity:
            print(f"   {precision}: passed={parity['passed']} matched={parity['matched']}/"
                  f"{parity['reference_detections']} min_iou={parity['min_iou']} "
                  f"mean_iou={parity['mean_iou']} max_conf_delta={parity['max_confidence_delta']} "
                  f"classes={parity['class_agreement']}")
        else:
            print(f"   {precision}: {result.get('status')} {result.get('reason', '')}")
    print('\ncontract semantics:', summary['contract_semantics']['conclusion'])
    print('int8:', summary['int8']['status'], '-', summary['int8']['reason'])
    print('status patch:', summary.get('status_patch'))
    print('precision policy:', json.dumps(summary.get('precision_policy'), indent=2))
    probes = (summary.get('onnx_provider_probes') or {})
    print('provider probes:', {key: value.get('state') for key, value in probes.items()})
    excluded = summary.get('non_authoritative_precisions') or []
    if excluded:
        print('EXCLUDED from the authoritative table, label not backed by engine dtypes:',
              [row['precision'] for row in excluded])
    graph_parity = summary.get('fp32_vs_mixed_fp16_onnx_parity')
    if graph_parity:
        print('fp32 -> mixed fp16 ONNX parity: passed=', graph_parity['passed'],
              'matched=', graph_parity['matched'], '/', graph_parity['reference_detections'],
              'min_iou=', graph_parity['min_iou'])
elif blocked_path:
    print(json.dumps(json.loads(blocked_path.read_text()), indent=2)[:2500])
else:
    print('no evidence written — inspect the previous cell output')

## 6. Package the evidence for download

Both runs are merged under `final/fp32` and `final/true_fp16`, alongside the conversion
record and `mixed_fp16_graph.json`, so the returned archive maps onto the repository
layout directly. Each run contributes its own environment, benchmark, parity, contract
semantics, `GPU_RUN_RESULT.md` and `status-patch.json` — the exact documentation edits a
verified run authorises.

Engine files stay out: a TensorRT engine is specific to the GPU, driver and TensorRT
version that built it, so the evidence references it by SHA-256 instead.

In [ ]:
import shutil

EXPECTED = ['environment.json', 'benchmark.json', 'parity_fp32.json', 'parity_fp16.json',
            'contract-semantics.json', 'tensorrt-verification.json', 'status-patch.json',
            'GPU_RUN_RESULT.md', 'samples.json', 'mixed_fp16_graph.json',
            'fp32_onnx_vs_mixed_fp16_onnx.json', 'onnx_cuda_baseline.json']
present = sorted(str(path.relative_to(EVIDENCE)) for path in EVIDENCE.rglob('*') if path.is_file())
print('evidence dir', EVIDENCE)
print('present     ', present)
missing = [name for name in EXPECTED if not any(entry.endswith(name) for entry in present)]
print('missing     ', missing)

base = pathlib.Path('/content') if IN_COLAB else pathlib.Path('/kaggle/working')
# Package the merged results only, so a stale local blocker can never travel next to a
# verified result in the same archive.
SOURCE = FINAL if FINAL.is_dir() and any(FINAL.iterdir()) else EVIDENCE
print('packaging   ', SOURCE)
archive = shutil.make_archive(str(base / 'visionops-tensorrt-evidence'), 'zip', str(SOURCE))
print('archive     ', archive, pathlib.Path(archive).stat().st_size, 'bytes')
print('\nRETURN THIS FILE TO THE CODING SESSION: visionops-tensorrt-evidence.zip')
print('(it lets docs/CURRENT_VERIFIED_STATE.md and docs/interview/ be updated from real '
      'measurements instead of from a claim)')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(archive)
    except Exception as error:
        print('download manually from the file browser:', error)
else:
    print('Kaggle: download it from the notebook Output panel (it is under /kaggle/working)')

## After the run

Report the status exactly, and no further than the evidence supports:

* `TensorRT on NVIDIA GPU: VERIFIED` — **only** if a precision reports `"passed": true` in
  its `parity_*.json` with a non-null `min_iou`.
* A Colab/Kaggle T4 is a real NVIDIA GPU and is **not** a Jetson. `Physical NVIDIA Jetson`,
  `JetPack`, Jetson thermal/power/NVDEC behaviour and any 10K physical fleet stay
  **NOT VERIFIED**. Do not blur that line.
* If the run was blocked, keep `blocked.json` and the failing records in
  `tensorrt-verification.json → steps`. A recorded blocker is more useful than a silent gap.
* `status-patch.json` lists only the edits the run authorises, with the exact line numbers
  it located, and a `must_not_change` list covering everything about Jetson and ARM64.
* A precision is **VERIFIED** only when the engine's own tensor dtypes back the label. If
  `precision_policy` excludes a precision, or a row shows up in
  `non_authoritative_precisions`, that number must never be quoted as that precision. An
  earlier run labelled an FP32 engine `fp16`; that result is kept, labelled, under
  `final/superseded/` for exactly that reason.
* True FP16 requires the engine to declare `DataType.HALF`, the converter to report
  `class_metadata_restored: true`, and the FP32-vs-FP16 ONNX parity check to have passed.
  Without all three, FP16 stays NOT VERIFIED.